# Regresión Logística Binaria — PhiUSIIL Phishing URL Dataset

## 1. Preparación del entorno

In [25]:
import numpy as np
import pandas as pd

from matplotlib import pyplot
from scipy import optimize

from sklearn.model_selection import train_test_split

%matplotlib inline
np.random.seed(42)

RUTA_DATASET = 'PhiUSIIL_Phishing_URL_Dataset.csv'

## 2. Carga y exploración del dataset

In [26]:
df = pd.read_csv(RUTA_DATASET)

print('Dimensiones del dataset completo:', df.shape)
df.head()

Dimensiones del dataset completo: (235795, 56)


### 2.1 Curación básica

In [27]:
duplicados = df.duplicated().sum()
print('Filas duplicadas encontradas:', duplicados)

if duplicados > 0:
    df = df.drop_duplicates().reset_index(drop=True)
    print('Nuevas dimensiones tras eliminar duplicados:', df.shape)

Filas duplicadas encontradas: 0


In [28]:
columnas_numericas = df.select_dtypes(include=[np.number]).columns
desvios = df[columnas_numericas].std()
columnas_constantes = desvios[desvios < 1e-6].index.tolist()

print('Columnas casi constantes encontradas:', columnas_constantes)

if columnas_constantes:
    df = df.drop(columns=columnas_constantes)
    print('Columnas restantes tras eliminar constantes:', df.shape[1])

Columnas casi constantes encontradas: []


In [29]:
X_raw = df.drop(columns=['label'])
y_raw = df[['label']]

In [30]:
print(df.dtypes.value_counts())
print()
print('Valores nulos totales:', df.isnull().sum().sum())

int64      41
float64    10
str         5
Name: count, dtype: int64

Valores nulos totales: 0


In [31]:
print(df['label'].value_counts())
print()
print((df['label'].value_counts(normalize=True) * 100).round(2))

df['label'].value_counts().plot(kind='bar', color=['#d62728', '#2ca02c'])
pyplot.xticks([0, 1], ['Phishing (0)', 'Legítima (1)'], rotation=0)
pyplot.ylabel('Cantidad de registros')
pyplot.title('Distribución de clases - PhiUSIIL')
pass

label
1    134850
0    100945
Name: count, dtype: int64

label
1    57.19
0    42.81
Name: proportion, dtype: float64


## 3. Preprocesamiento con Pandas

In [32]:
X_raw = X_raw.drop(columns=['FILENAME'], errors='ignore')

X_df = X_raw.select_dtypes(include=[np.number]).copy()
columnas_texto = X_raw.columns.difference(X_df.columns).tolist()

print('Columnas de texto descartadas:', columnas_texto)
print()
print('Cantidad de características numéricas utilizadas (n):', X_df.shape[1])
print('Cantidad de ejemplos (m):', X_df.shape[0])

Columnas de texto descartadas: ['Domain', 'TLD', 'Title', 'URL']

Cantidad de características numéricas utilizadas (n): 50
Cantidad de ejemplos (m): 235795


In [33]:
print('Valores nulos por columna (solo columnas con al menos uno):')
nulos = X_df.isnull().sum()
print(nulos[nulos > 0])

print()
print('Valores infinitos:', np.isinf(X_df.to_numpy(dtype=float)).sum())

Valores nulos por columna (solo columnas con al menos uno):
Series([], dtype: int64)

Valores infinitos: 0


In [34]:
X = X_df.to_numpy(dtype=float)
y = y_raw['label'].to_numpy(dtype=float)

print('Dimensiones de X:', X.shape)
print('Dimensiones de y:', y.shape)

Dimensiones de X: (235795, 50)
Dimensiones de y: (235795,)


## 4. División 80% / 20%

In [35]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print('Entrenamiento:', X_train.shape, y_train.shape)
print('Prueba:       ', X_test.shape, y_test.shape)
print()
print('Proporción de clase 1 en entrenamiento: {:.2f}%'.format(y_train.mean() * 100))
print('Proporción de clase 1 en prueba:        {:.2f}%'.format(y_test.mean() * 100))

Entrenamiento: (188636, 50) (188636,)
Prueba:        (47159, 50) (47159,)

Proporción de clase 1 en entrenamiento: 57.19%
Proporción de clase 1 en prueba:        57.19%


## 5. Escalado de características

In [36]:
mu = X_train.mean(axis=0)
sigma = X_train.std(axis=0)

sigma[sigma == 0] = 1.0

X_train_norm = (X_train - mu) / sigma
X_test_norm = (X_test - mu) / sigma

print('Media (primeras 5 columnas):', np.round(mu[:5], 3))
print('Desvío (primeras 5 columnas):', np.round(sigma[:5], 3))

Media (primeras 5 columnas): [3.4584e+01 2.1475e+01 3.0000e-03 7.8419e+01 8.4500e-01]
Desvío (primeras 5 columnas): [40.978  9.167  0.053 28.99   0.217]


In [37]:
m_train, n = X_train_norm.shape
X_train_norm = np.concatenate([np.ones((m_train, 1)), X_train_norm], axis=1)

m_test = X_test_norm.shape[0]
X_test_norm = np.concatenate([np.ones((m_test, 1)), X_test_norm], axis=1)

print('Nueva dimensión de X_train con intercepción:', X_train_norm.shape)
print('Nueva dimensión de X_test con intercepción: ', X_test_norm.shape)

Nueva dimensión de X_train con intercepción: (188636, 51)
Nueva dimensión de X_test con intercepción:  (47159, 51)


## 6. Implementación del modelo

### 6.1 Función sigmoidea

In [38]:
def sigmoid(z):
    z = np.array(z)
    z = np.clip(z, -500, 500)
    return 1 / (1 + np.exp(-z))

print('sigmoid(0) =', sigmoid(0))

sigmoid(0) = 0.5


### 6.2 Función de costo y gradiente

In [39]:
def costFunction(theta, X, y):
    m = y.size

    h = sigmoid(X.dot(theta.T))
    h = np.clip(h, 1e-12, 1 - 1e-12)

    J = (1 / m) * np.sum(-y * np.log(h) - (1 - y) * np.log(1 - h))
    grad = (1 / m) * (h - y).dot(X)

    return J, grad

In [40]:
initial_theta = np.zeros(n + 1)
cost, grad = costFunction(initial_theta, X_train_norm, y_train)

print('Costo en theta inicial (ceros): {:.6f}'.format(cost))

Costo en theta inicial (ceros): 0.693147


## 7. Entrenamiento por descenso de gradiente

In [41]:
def descensoGradiente(theta, X, y, alpha, num_iters):
    theta = theta.copy()
    J_history = []

    for i in range(num_iters):
        cost, grad = costFunction(theta, X, y)
        theta = theta - alpha * grad
        J_history.append(cost)

    return theta, J_history

In [42]:
alpha = 0.5
num_iters = 1500

theta = np.zeros(n + 1)
theta, J_history = descensoGradiente(theta, X_train_norm, y_train, alpha, num_iters)

pyplot.plot(np.arange(len(J_history)), J_history, lw=2)
pyplot.xlabel('Número de iteraciones')
pyplot.ylabel('Costo J(theta)')
pyplot.title('Convergencia del descenso de gradiente')
pass

print('Costo final tras {} iteraciones: {:.6f}'.format(num_iters, J_history[-1]))

Costo final tras 1500 iteraciones: 0.002892


## 8. Ajuste fino con `scipy.optimize.minimize`

In [43]:
options = {'maxiter': 400}

res = optimize.minimize(costFunction,
                         theta,
                         (X_train_norm, y_train),
                         jac=True,
                         method='BFGS',
                         options=options)

theta_opt = res.x

print('¿Convergió?:', res.success)
print('Mensaje:', res.message)
print('Costo con theta optimizado: {:.6f}'.format(res.fun))

¿Convergió?: True
Mensaje: Optimization terminated successfully.
Costo con theta optimizado: 0.000034


## 9. Evaluación del modelo

### 9.1 Función de predicción

In [44]:
def predict(theta, X):
    p = sigmoid(X.dot(theta.T))
    return np.round(p)

### 9.2 Exactitud en entrenamiento y prueba

In [45]:
p_train = predict(theta_opt, X_train_norm)
acc_train = np.mean(p_train == y_train) * 100

p_test = predict(theta_opt, X_test_norm)
acc_test = np.mean(p_test == y_test) * 100

print('Exactitud en entrenamiento: {:.2f} %'.format(acc_train))
print('Exactitud en prueba:        {:.2f} %'.format(acc_test))

Exactitud en entrenamiento: 100.00 %
Exactitud en prueba:        100.00 %


### 9.3 Matriz de confusión y métricas adicionales

In [46]:
def matriz_confusion(y_true, y_pred):
    VP = np.sum((y_pred == 1)  & (y_true == 1))
    VN = np.sum((y_pred == 0)  & (y_true == 0))
    FP = np.sum((y_pred == 1)  & (y_true == 0))
    FN = np.sum((y_pred == 0)  & (y_true == 1))
    return VP, VN, FP, FN

VP, VN, FP, FN = matriz_confusion(y_test, p_test)

precision = VP / (VP + FP)
recall = VP / (VP + FN)
f1 = 2 * precision * recall / (precision + recall)

print('Matriz de confusión (conjunto de prueba):')
print('                  Predicho 0   Predicho 1')
print('Real 0 (phishing)   {: >8d}     {: >8d}'.format(VN, FP))
print('Real 1 (legítima)   {: >8d}     {: >8d}'.format(FN, VP))
print()
print('Precisión: {:.4f}'.format(precision))
print('Recall:    {:.4f}'.format(recall))
print('F1-score:  {:.4f}'.format(f1))

Matriz de confusión (conjunto de prueba):
                  Predicho 0   Predicho 1
Real 0 (phishing)      20189            0
Real 1 (legítima)          1        26969

Precisión: 1.0000
Recall:    1.0000
F1-score:  1.0000


In [47]:
matriz = np.array([[VN, FP], [FN, VP]])

fig, ax = pyplot.subplots()
im = ax.imshow(matriz, cmap='Blues')

for i in range(2):
    for j in range(2):
        ax.text(j, i, matriz[i, j], ha='center', va='center', color='black', fontsize=14)

ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(['Predicho 0 (phishing)', 'Predicho 1 (legítima)'])
ax.set_yticklabels(['Real 0 (phishing)', 'Real 1 (legítima)'])
ax.set_title('Matriz de confusión - Conjunto de prueba')
pyplot.colorbar(im)
pass

### 9.4 Ejemplo de predicción individual

In [48]:
idx = 0
prob_ejemplo = sigmoid(np.dot(X_test_norm[idx], theta_opt))

print('Probabilidad estimada de que el registro sea una URL legítima: {:.4f}'.format(prob_ejemplo))
print('Clase predicha:', int(np.round(prob_ejemplo)))
print('Clase real:    ', int(y_test[idx]))

Probabilidad estimada de que el registro sea una URL legítima: 1.0000
Clase predicha: 1
Clase real:     1


## 10. Conclusiones

- Dataset: 235.795 ejemplos, 50 características numéricas.
- División estratificada 80/20.
- Escalado de características (estandarización) sobre el conjunto de entrenamiento.
- Modelo implementado desde cero (sigmoidea, costo, gradiente, descenso de gradiente).
- Ajuste fino con `scipy.optimize.minimize` (método BFGS).
- Exactitud en prueba: 100.00%.
- Precisión, Recall y F1-score: 1.0000.